# **📘 Building a RAG Pipeline from Scratch........**

This notebook demonstrates the **end-to-end implementation of a Retrieval-Augmented Generation (RAG) pipeline** built from scratch using modern NLP and vector database tools.

The goal of this notebook is to show how **unstructured PDF data** can be ingested, processed, embedded, and stored in a vector database to enable **semantic search and context-aware question answering**.

---

## **🔍 What This Notebook Covers**

- 📄 **Document Loading**  
  Loading PDF documents using LangChain document loaders.

- ✂️ **Text Chunking**  
  Splitting large documents into smaller, overlapping chunks for better retrieval performance.

- 🧠 **Embedding Generation**  
  Converting text chunks into dense vector embeddings using **Sentence Transformers**.

- 🗄️ **Vector Store Creation**  
  Storing embeddings in **ChromaDB** for fast similarity search.

- ⚙️ **Modular Pipeline Design**  
  Clean, reusable, and scalable components suitable for production-grade RAG systems.

---

## **🚀 Why Retrieval-Augmented Generation (RAG)?**

Large Language Models (LLMs) are limited by static training data and can hallucinate.  
RAG enhances LLMs by **retrieving relevant context from external documents** before generating responses.

This approach is widely used in:
- Document-based Question Answering
- Chatbots over PDFs
- Enterprise knowledge assistants
- Search and recommendation systems

---

## **🧩 Technologies Used**

- Python  
- LangChain (latest modular packages)  
- SentenceTransformers  
- ChromaDB  
- PyPDF  
- NumPy  

---

## **🎯 Outcome**

By the end of this notebook, you will have:
- A populated **vector database**
- Embedded and indexed PDF documents
- A strong foundation to extend this into a **full RAG-based QA application**

> **Note:**  
> This notebook uses updated LangChain package structure (2025 compatible), which may differ from older tutorials.


#### **Data Ingestion**

#### Creating MetaData
>This is use to filter the Documents 

In [1]:
from langchain_core.documents import Document

In [2]:
# Creating an metadata
doc = Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source" : "example.txt",
        "pages" : 1,
        "author" : "Sujal Warghe",
        "date_created" : "2025-01-01"}
)
doc
# metadata is use for when someone asked about rag by sujal warghe the the llm does not have
# to go through all the pdfs and documents it directly comes to these pdf
# using these filter

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Sujal Warghe', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [3]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
# in this cell we are creating an text files with the content through coding
sample_texts={
    "../data/text_files/python_intro.txt":"""Introduction to Python

Python is a high-level, interpreted, and general-purpose programming language. It was created by Guido van Rossum and released in 1991. Python is designed with an emphasis on code readability and simplicity, which makes it easy to learn and use, especially for beginners.

Python uses a clear and English-like syntax, allowing programmers to write fewer lines of code compared to many other programming languages. It is a platform-independent language, meaning Python programs can run on Windows, Linux, and macOS without any modification.

Python is an open-source language and has a large collection of libraries and frameworks. Due to its flexibility and power, Python is widely used in various fields such as web development, data science, machine learning, artificial intelligence, automation, and scientific computing.

Features of Python

Simple and easy to learn

Interpreted language

Platform independent

Open source

Large standard library

Object-oriented programming support

Applications of Python

Web Development (Django, Flask)

Data Science and Data Analysis

Machine Learning and Artificial Intelligence

Automation and Scripting

Game Development

Desktop Applications

Conclusion

Python is a powerful, flexible, and user-friendly programming language. Its simplicity and wide range of applications make it one of the most popular programming languages in the world today.""",

    "../data/text_files/machine_learning.txt":"""Introduction to Machine Learning

Machine Learning is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve their performance without being explicitly programmed. Instead of writing fixed rules, machine learning systems identify patterns in data and make decisions or predictions based on those patterns.

Machine learning works by using algorithms that are trained on large amounts of data. During training, the model learns relationships between input data and output results. Once trained, the model can make predictions or classifications on new and unseen data.

Machine learning is widely used in many real-world applications such as spam email filtering, recommendation systems, face recognition, medical diagnosis, fraud detection, and self-driving cars.

Types of Machine Learning

Supervised Learning
The model is trained using labeled data (input with correct output).
Example: Linear Regression, Decision Tree, KNN

Unsupervised Learning
The model works with unlabeled data and finds hidden patterns.
Example: K-Means Clustering, Apriori Algorithm

Reinforcement Learning
The model learns by interacting with the environment and receiving rewards or penalties.
Example: Game-playing AI, Robotics

Key Components of Machine Learning

Data

Features

Algorithm

Model

Training and Testing

Evaluation

Advantages of Machine Learning

Can handle large amounts of data

Improves performance over time

Reduces human effort

Useful for complex problem solving

Conclusion

Machine learning is a powerful technology that allows systems to learn automatically from data and make intelligent decisions. It plays an important role in modern technology and is widely used across various industries."""
}

for filepath,content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("😊 Sample text Files created")


😊 Sample text Files created


WE created two text files now we are going to read athat text files through the text_loader

In [5]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8") 
document=loader.load()
print(document)

c:\Rag_Developement\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Introduction to Python\n\nPython is a high-level, interpreted, and general-purpose programming language. It was created by Guido van Rossum and released in 1991. Python is designed with an emphasis on code readability and simplicity, which makes it easy to learn and use, especially for beginners.\n\nPython uses a clear and English-like syntax, allowing programmers to write fewer lines of code compared to many other programming languages. It is a platform-independent language, meaning Python programs can run on Windows, Linux, and macOS without any modification.\n\nPython is an open-source language and has a large collection of libraries and frameworks. Due to its flexibility and power, Python is widely used in various fields such as web development, data science, machine learning, artificial intelligence, automation, and scientific computing.\n\nFeatures of Python\n\nSimple and easy to learn\n\nInterpre

##### **Reading .txt Files from Directory**

In [6]:
### Directory Loader
# this is used for oading all the text files from the directory
from langchain_community.document_loaders import DirectoryLoader

## load all the text files  from the directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls=TextLoader, #loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Introduction to Machine Learning\n\nMachine Learning is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve their performance without being explicitly programmed. Instead of writing fixed rules, machine learning systems identify patterns in data and make decisions or predictions based on those patterns.\n\nMachine learning works by using algorithms that are trained on large amounts of data. During training, the model learns relationships between input data and output results. Once trained, the model can make predictions or classifications on new and unseen data.\n\nMachine learning is widely used in many real-world applications such as spam email filtering, recommendation systems, face recognition, medical diagnosis, fraud detection, and self-driving cars.\n\nTypes of Machine Learning\n\nSupervised Learning\nThe model is trained using labeled data (input

##### **Reading pdf files from Directory**

In [7]:
# This code is use for loading .pdf files from the Directory  

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files
    loader_cls=PyMuPDFLoader, #loader class to use
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2020-08-19T20:01:58-05:00', 'source': '..\\data\\pdf\\01-ml-overview__notes.pdf', 'file_path': '..\\data\\pdf\\01-ml-overview__notes.pdf', 'total_pages': 22, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-08-19T20:01:58-05:00', 'trapped': '', 'modDate': "D:20200819200158-05'00'", 'creationDate': "D:20200819200158-05'00'", 'page': 0}, page_content='STAT 451: Introduction to Machine Learning\nLecture Notes\nSebastian Raschka\nDepartment of Statistics\nUniversity of Wisconsin–Madison\nhttp://stat.wisc.edu/∼sraschka/teaching/stat451-fs2020/\nFall 2020\nContents\n1\nL01: What is Machine Learning? An Overview.\n1\n1.1\nMachine Learning – The Big Picture . . . . . . . . . . . . . . . . . . . . . . .\n1\n1.2\nApplications of Machine Learning . . . . . . . . . . . . . . . . . . . . . . . . .\n3\n1.3\nOverview of the Categories of Machine Learning\

Task : how to read excel, db 
go to langchain document _loaders

##### **Reading .csv files from Directory**

In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = DirectoryLoader(
    path="../data/csv",
    glob="**/*.csv",
    loader_cls=CSVLoader,
    show_progress=False
)

csv_documents = loader.load()

print(f"Total documents loaded: {len(csv_documents)}")
print(csv_documents[45])


Total documents loaded: 160
page_content=': 35
sepal_length: 5.0
sepal_width: 3.2
petal_length: 1.2
petal_width: 0.2
species: setosa' metadata={'source': '..\\data\\csv\\expo-flower.csv', 'row': 35}


In [9]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import UnstructuredExcelLoader

loader = DirectoryLoader(
    path="../data/excel",
    glob="**/*.xlsx",
    loader_cls=UnstructuredExcelLoader,
    show_progress=False
)

excel_documents = loader.load()
print(len(excel_documents))
print(excel_documents[0])


2
page_content='Sr No College Code College Name Branch City Percentile 1 I 9698\n(95.4813409) 16599\n(91.9454241) 28642\n(83.7510773) 5336\n(97.5366265) 2 I 3389\n(98.4034964) 9987\n(95.3361384) 25392\n(86.3687058) 2601\n(98.7641491) 3 I 6384\n(97.0514111) 14928\n(92.8495905) 47901\n(59.9367772) 4136\n(98.0761571) 4 I 9970\n(95.3430183) 14941\n(92.8412621) 34814\n(77.7072485) 6238\n(97.1216587) 5 I 9525\n(95.5613653) 18302\n(90.9854942) 53515\n(50.2596265) 6201\n(97.1372291) 6 I 13660\n(93.5194774) 25951\n(85.9439612) 54048\n(49.2624002) 11071\n(94.8226791) 7 I 7194\n(96.6722913) 14267\n(93.2066221) 33295\n(79.3171498) 5197\n(97.6014426) 8 I 21194\n(89.2716699) 23289\n(87.8844608) 56941\n(44.0144696) 19086\n(90.5419204) 9 I 14456\n(93.1110274) 62928\n(33.1177625) 15623\n(92.4730054) 15623\n(92.4730054) 10 I 32067\n(80.5367062) 32070\n(80.5352578) 20286\n(89.8329266) 40015\n(71.4389172) 11 I 58606\n(40.9195918) 61102\n(36.2705764) 60229\n(37.9108942) 46374\n(62.2741395) 12 I 32029\n(80.

##### **Embedding And VectorDB**

31:15

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        
        except Exception as e:
            print(f"Error Loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
            
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    
## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F519B81F50>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: c8a48415-d4e8-4a31-8f6a-4e74e9b641a3)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading embedding model: all-MiniLM-L6-v2


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F519B81210>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 9536558b-5128-47d2-b7b8-e5d9990cb947)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F519B83D10>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 9f102576-90d5-4cc7-b526-90a12134bf28)')' thrown while requesting HEAD https://huggingface.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs


In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
pdf_loader = PyPDFDirectoryLoader("../data/pdf")
all_pdf_documents = pdf_loader.load()
chunks = split_documents(all_pdf_documents)
chunks

Split 107 documents into 234 chunks


[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2020-08-19T20:01:58-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2020-08-19T20:01:58-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '..\\data\\pdf\\01-ml-overview__notes.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}, page_content='STAT 451: Introduction to Machine Learning\nLecture Notes\nSebastian Raschka\nDepartment of Statistics\nUniversity of Wisconsin–Madison\nhttp://stat.wisc.edu/∼sraschka/teaching/stat451-fs2020/\nFall 2020\nContents\n1 L01: What is Machine Learning? An Overview. 1\n1.1 Machine Learning – The Big Picture . . . . . . . . . . . . . . . . . . . . . . . 1\n1.2 Applications of Machine Learning . . . . . . . . . . . . . . . . . . . . . . . . . 3\n1.3 Overview of the Categories of Machine Learning . . . . . . . . . . . . . . . . 4

#### **VectorStore......**

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        intialize the vector store
        
        Args:
           collection_name: Name of the ChromaDB collection
           persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_direcory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        """
        initialize chromaDB client and collection
        """
        try:
            # Create persistant ChromaDB client
            os.makedirs(self.persist_direcory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_direcory)

            # Get or create collections
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """ 
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of Langchain documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents to vector of embeddings")
        
        print(f"Adding {len(documents)} documents must match of embeddings")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 702


In [ ]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2020-08-19T20:01:58-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2020-08-19T20:01:58-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '..\\data\\pdf\\01-ml-overview__notes.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}, page_content='STAT 451: Introduction to Machine Learning\nLecture Notes\nSebastian Raschka\nDepartment of Statistics\nUniversity of Wisconsin–Madison\nhttp://stat.wisc.edu/∼sraschka/teaching/stat451-fs2020/\nFall 2020\nContents\n1 L01: What is Machine Learning? An Overview. 1\n1.1 Machine Learning – The Big Picture . . . . . . . . . . . . . . . . . . . . . . . 1\n1.2 Applications of Machine Learning . . . . . . . . . . . . . . . . . . . . . . . . . 3\n1.3 Overview of the Categories of Machine Learning . . . . . . . . . . . . . . . . 4

In [ ]:
### convert the text  to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## store int he vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 234 texts...


Batches: 100%|██████████| 8/8 [00:10<00:00,  1.33s/it]


Generated embeddings with shape: (234, 384)
Adding 234 documents must match of embeddings
Successfully added 234 documents to vector store
Total documents in collection: 936


#### **Retrieval Pipeline From VectorStore**

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retirever

        Args:
           vector_store: vector containing document embeddings
           embedding_manager for generating query embeddings 
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
       Retrieve relevant documents for a query

       Args:
          query: The search query
          top_k: Number of top results to return
          score_threshold: Minimum similarity score threshold

       Return:
           List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i + 1
                        })

                print(f'Retrieved {len(retrieved_docs)} documents (after filtering)')
            else:
                print("No documents found")

            return retrieved_docs
        except Exception as e:
            print(f'Error during retrieval: {e}')
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)




In [ ]:
rag_retriever.retrieve("Type of machine learnings")

Retrieving documents for query: Type of machine learnings
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_94d38b25_138',
  'content': 'Aditya Engineering College (A)   4 \n \nP.MURALI Assistant Professor CSE Department \n \n \n \nTOPIC-2: Types of Machine Learning Systems',
  'metadata': {'source': '..\\data\\pdf\\ML UNIT-1 NOTES.pdf',
   'author': 'Mounika',
   'creationdate': '2023-05-19T06:55:36+00:00',
   'moddate': '2023-05-19T06:55:37+00:00',
   'content_length': 131,
   'producer': 'www.ilovepdf.com',
   'page': 3,
   'total_pages': 20,
   'page_label': '4',
   'creator': 'Microsoft® Word 2016',
   'doc_index': 138},
  'similarity_score': 0.5247847735881805,
  'distance': 0.47521522641181946,
  'rank': 1},
 {'id': 'doc_227f9ee0_138',
  'content': 'Aditya Engineering College (A)   4 \n \nP.MURALI Assistant Professor CSE Department \n \n \n \nTOPIC-2: Types of Machine Learning Systems',
  'metadata': {'doc_index': 138,
   'page_label': '4',
   'content_length': 131,
   'producer': 'www.ilovepdf.com',
   'moddate': '2023-05-19T06:55:37+00:00',
   'page': 3,
   'total_pages

In [ ]:
rag_retriever.retrieve("Why Artificial Intelligence?")

Retrieving documents for query: Why Artificial Intelligence?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.32it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_092f2605_134',
  'content': '• Here, one of the booming technologies of computer science is Artificial Intelligence which is ready \nto create a new revolution in the world by making intelligent machines. \n• Artificial Intelligence is composed of two words Artificial and Intelligence, where Artificial defines \n"man-made," and intelligence defines "t hinking power", hence AI means "a man -made thinking \npower." \n• So, we can define AI as:  "It is a branch of computer science by which we can create intelligent \nmachines which can behave like a human, think like humans, and able to make decisions."  \n• Artificial Intelligence exists when a machine can have human based skills such as learning, \nreasoning, and solving problems. \nWhy Artificial Intelligence? \n• With the help of AI, you can create such software or devices which can solve real -world problems \nvery easily and with accuracy such as health issues, marketing, traffic issues, etc.',
  'metadata': {'creator':

### **Integration Vectordb Context pipeline with LLM output**

#### *Simple RAG Pipeline with Groq LLM*

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please set it before running the code.")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever,llm,top_k=3):
    ## retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context to answer the question concisely."
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}
        
        Question: {query}
        
        Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [ ]:
answer = rag_simple("Types of machine learning?", rag_retriever,llm)
print(answer)

Retrieving documents for query: Types of machine learning?
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


There are three main types of machine learning systems:

1. Supervised Learning
2. Unsupervised Learning
3. Reinforcement Learning


#### **Enhanced RAG Pipeline Features**

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer' : 'No relevent context found.', 'sources' : [], 'confidence' : 0.0, 'context' : ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answwer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer' : response.content,
        'source' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Which is the Beutiful place in India?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: Which is the Beutiful place in India?
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevent context found.
Sources: []
Confidence: 0.0
Context Preview: 


### Advanced RAG Pipeline

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Why Artificial Intelligence?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: Why Artificial Intelligence?
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
• Here, one of the booming technologies of computer science is Artificial Intelligence which is ready 
to create a new revolution in the world by making intelligent machin

es. 
• Artificial Intelligence is composed of two words Artificial and Intelligence, where Artificial defines 
"man-made," and intelligence defines "t hinking power", hence AI means "a man -made thinking 
power." 
• So, we can define AI as:  "It is a branch of computer science by which we can create intelligent 
machines which can behave like a human, think like humans, and able to make decisions."  
• Artificial Intelligence exists when a machine can have human based skills such as learning, 
reasoning, and solving problems. 
Why Artificial Intelligence? 
• With the help of AI, you can create such software or devices which can solve real -world problems 
very easily and with accuracy such as health issues, marketing, traffic issues, etc.

• Here, one of the booming technologies of computer science is Artificial Intelligence which is ready 
to create a new revolution in the world by making intelligent machines. 
• Artificial Intelligence is composed of two words Artificial and Intellig